# The HALD Knowledge Graph

This section has been inspired by the work of Robert Haas on Biomedical Knowledge Graphs which can be found at 
https://github.com/robert-haas/awesome-biomedical-knowledge-graphs/tree/main
The source of the data is the webpage on [Figshare](https://figshare.com/articles/dataset/HALD_a_human_aging_and_longevity_knowledge_graph_for_precision_gerontology_and_geroscience_analyses/22828196). This include versions in JSON and CSV. The CSV versions are structured for a particular package which we will not be using, so we will use the json packages which are more general.

First we'll create a data directory.

In [1]:
import os
Download = False
datadir = "HALD_Dataset"
if not os.path.exists(datadir):
    os.mkdir(datadir)

Now we define the list of the files that we want to download. We'll define a *list* of *tuples*, with each tuple representing one of the files that we want to fetch, specifying three things:

* The name that we want the file to be called.
* The URL from where it will be downloaded.
* The MD5 checksum which will allow us to verify the downloaded file's integrity.

In [2]:
# List of files to download
filelist = [
    ("Entity_info.json", "https://figshare.com/ndownloader/files/43612509", '1746cde24a1bac0460f1ccf646608cc9'),
    ("Literature_Info.json", "https://figshare.com/ndownloader/files/43612512", "10b78e8ec30f5b85f2a58d8fe24f056b"),
    ("Longevity_Biomarkers.json", "https://figshare.com/ndownloader/files/43612497", "0dbd9c3f8474dc3cd744ed38af460d75"),
    ("Relation_Info.json", "https://figshare.com/ndownloader/files/43612506", "0c1fa199269adc58f64ad4d5b9fd87b9"),
    ("Aging_Biomarkers.json", "https://figshare.com/ndownloader/files/43612503", "abd0eb6cb7295ae500c5d676b7797324")
]

Now we can download the files. For each file in `filelist` we will:

* Download the file from the URL.
* If the download request indicates that the download is unsuccessful, print an error.
* If the download is successfull, verify the checksum and if that is correct, write the file to disk in `datadir`

In [3]:
import requests
import hashlib

if Download:
    for f in filelist:
        response = requests.get(f[1])
        file_Path = datadir + "/" + f[0]
        if response.status_code != 200:
            print('Failed to download file {f[0]} from {f[1]}')
        else:
            m = hashlib.md5()
            m.update(response.content)
            if m.hexdigest() == f[2]:
                print(f"SUCCESS: File {f[0]} downloaded from {f[1]} with correct checksum {f[2]}")
                with open(file_Path, 'wb') as file:
                    file.write(response.content)
            else:
                print(f"ERROR: File {f[0]} downloaded from {f[1]} with incorrect checksum {m.hexdigest()} (should be {f[2]})")            


## What does the data look like?

Let us inspect these files. The two key files here are those containing the *entities* (nodes) and the *edges* (relations).

In [4]:
import json

def load_json(fname):
    with open(fname, 'rb') as file:
        return json.load(file)

Entity_info = load_json(f"{datadir}/{filelist[0][0]}")
Literature_info = load_json(f"{datadir}/{filelist[1][0]}")
Longevity_Biomarkers = load_json(f"{datadir}/{filelist[2][0]}")
Relation_info = load_json(f"{datadir}/{filelist[3][0]}")
Aging_Biomarkers = load_json(f"{datadir}/{filelist[4][0]}")

In [5]:
Entity_info['rs1556516']

[{'entity': 'rs1556516',
  'type': 'Mutation',
  'PMID': ['33720087'],
  'official full name': None,
  'sentence': [['METHODS: We evaluated the impact of 10 SNPs involved in both human liver/metabolic diseases and healthspan (interleukin-6 [IL-6] rs1800795, antisense non coding RNA in the INK4 locus (ANRIL) rs1556516, SH2B3/ATXN2 rs7137828, FURIN rs17514846, TP53 rs1042522, APOC3 rs2542052, KL rs9536314, KL rs9527025, SIRT6 rs107251, FOXO3 rs2802292) on NAFLD-related metabolic and liver features in 177 pediatric patients with biopsy-proven NAFLD, by comparing them to 146 healthy controls.',
    'By testing potential synergies using the MDR approach, the best combination to diagnose NAFLD (P = 0.0011) resulted in the one encompassing IL-6 rs1800795 and ANRIL rs1556516.',
    'CONCLUSION: In conclusion, here we demonstrated a synergic interaction between IL-6 rs1800795 and ANRIL rs1556516 in the diagnosis of NAFLD, and NAFLD-associated hyperglycemia in children.']],
  'numbers of article

In [6]:
Entity_info['MLH1']

[{'entity': 'MLH1',
  'type': 'Gene',
  'PMID': ['12612901',
   '30275527',
   '25311944',
   '22936446',
   '19949675',
   '22406557',
   '23240038',
   '11325821',
   '21042749',
   '25556597',
   '17556535',
   '29425284',
   '22740444',
   '10954253',
   '37380216'],
  'official full name': 'mutL homolog 1',
  'sentence': [['Most such cancers have the CpG island methylator phenotype (CIMP+) with methylation and transcriptional silencing of the mismatch repair gene MLH1.'],
   ['Our group recently demonstrated that aging human HSCs accumulate microsatellite instability coincident with loss of MLH1, a DNA Mismatch Repair (MMR) protein, which could reasonably predispose to radiation-induced HSC malignancies.',
    'In addition, whole-exome sequencing analysis revealed high SNVs and INDELs in lymphomas being driven by loss of Mlh1 and frequently mutated genes had a strong correlation with human leukemias.'],
   ['ARID1A loss was observed in 9% (22/257) of the cohort: 24% of MMR-deficie

There are some troublesome entries in this dataset. We need to go through all of the entries and clear up any names or datatypes that are not legal. This means:

* Names containing characters that are not `[a-zA-Z0-9]`
* Names beginning with `[0-9]`
* Names that are reserved in Python: `type`
* Data types that are not `[str, int, float, bool]`

In [7]:
# Define a function to fix
# * Names containing characters that are not `[a-zA-Z0-9]`
# * Names beginning with `[0-9]`

import re
def fix_name(s):
    s = re.sub('[^0-9a-zA-Z]+', '_', s)
    if re.match('^[0-9]',s):
        s = f"X{s}"
    return s

fix_name("12-abc")

'X12_abc'

In [8]:
# First check the keys of the entities and relations themselves

def correct_keys(d):
    corrected_keys = dict()
    for k in d.keys():
        new_key = fix_name(k)
        if new_key != k:
            corrected_keys[k] = new_key
    for k in corrected_keys.keys():
        d[corrected_keys[k]] = d.pop(k,None)    
    return d

Entity_info = correct_keys(Entity_info)
Relation_info = correct_keys(Relation_info)

Now correct the keys of the dictionaries within `Entity_info` and `Relation_info`. Also fix entities with known troublesome names (entity[type]) and collect attributes with type not in `[int, float, str, bool]`

Potentially also need to rename Relation_info[*]['relationship] = "name"

In [9]:
Entity_info.pop('rs1556516',None)
Entity_info['DiseaseEntity'] = Entity_info.pop('Disease',None)
Entity_info['DiseaseEntity'][0]['entity'] = 'DiseaseEntity'
Relation_info.pop('Non-alcoholic Fatty Liver Disease-rs1556516-CDKN2B-AS1', None)


for k in Entity_info.keys():
    Entity_info[k][0]['entitytype'] = fix_name(Entity_info[k][0].pop('type',None))
    Entity_info[k][0] = correct_keys(Entity_info[k][0])
    attributes_for_deletion = [i for i in Entity_info[k][0].keys() if type(Entity_info[k][0][i]) not in [str,int,float,bool]]
    for i in attributes_for_deletion:
        Entity_info[k][0].pop(i,None)

for k in Relation_info.keys():
    
    Relation_info[k] = correct_keys(Relation_info[k])
    # Fix all of the relations that get used to create attributes
    Relation_info[k]['relationship'] = fix_name(Relation_info[k]['relationship'])
    if Relation_info[k]['relationship'] == 'name':
        Relation_info[k]['relationship'] = 'has_name'
    Relation_info[k]['source_type'] = [fix_name(i) for i in Relation_info[k]['source_type']]
    Relation_info[k]['target_type'] = [fix_name(i) for i in Relation_info[k]['target_type']]
    Relation_info[k]['source_entity'] = fix_name(Relation_info[k]['source_entity'])
    Relation_info[k]['target_entity'] = fix_name(Relation_info[k]['target_entity'])
    if Relation_info[k]['source_entity'] == 'Disease':
        Relation_info[k]['source_entity'] = 'DiseaseEntity'
    if Relation_info[k]['target_entity'] == 'Disease':
        Relation_info[k]['target_entity'] = 'DiseaseEntity'
    attributes_for_deletion = []
    for i in Relation_info[k].keys():
        if type(Relation_info[k][i]) not in [str,int,float,bool] and i not in ['source_type','target_type']:
            attributes_for_deletion.append(i)
    
    for i in attributes_for_deletion:
        Relation_info[k].pop(i,None)



So each entry is a list of length 1, and that entry contains a dictionary of the node's attributes. Those attributes are sometimes lists.

Let's get a list of the types of the entities create them. Whilst we are doing this, we rename one of the types as it conflicts with a Python keyword

In [10]:
from owlready2 import *
onto = get_ontology("http://www.dummy.info/new.owl")

EntityTypes = set()

for k in Entity_info.keys():
    EntityTypes.add(Entity_info[k][0]['entitytype'])

with onto:
    EntityClasses = dict()
    for entity in EntityTypes:
        EntityClasses[entity] = type(entity, (Thing,), dict())
        print(f'Created entity class {EntityClasses[entity]}')


Created entity class new.Protein
Created entity class new.Peptide
Created entity class new.Mutation
Created entity class new.Lipid
Created entity class new.Disease
Created entity class new.Pharmaceutical_Preparations
Created entity class new.Carbohydrate
Created entity class new.RNA
Created entity class new.Gene
Created entity class new.Toxin


Now we do the same for the relationships. We do this now because we will want to attach attributes to them so we need to create all the attributes in one go.

In [11]:
# Remove a trouble relation from the dataset. For reasons unknown, the relationship type "rs1556516" does not play well with owlready2

# First fetch all the types of relation and  create the ObjectProperties
with onto:
    RelationType = dict()
    for relation in Relation_info.keys():
        Domain = Relation_info[relation]['source_type']
        Range = Relation_info[relation]['target_type']
        DomainClasses = {EntityClasses[i] for i in Domain}
        RangeClasses = {EntityClasses[i] for i in Range}
        RelationType[Relation_info[relation]['relationship']] = {'domain': list(DomainClasses), 'range': list(RangeClasses)}

    RelationClasses = dict()
    for relation in RelationType.keys():
        RelationClasses[relation] = type(relation, (ObjectProperty,), dict())


Now we have the class for the entity and the relations, we can construct the unified set of attributes. We loop over the entities identifying new attributes and their domain and range. Here we need to be mindful of a limitation: Owlready2 does not support all datatypes as ranges for the DataProperty class. We will ignore those attributes. The list of valid types for the range can be found at https://owlready2.readthedocs.io/en/latest/properties.html

In [12]:
# First do the entities
EntityAttributeType = dict()

for entity in Entity_info.keys():
    for attribute in Entity_info[entity][0].keys():
        ClassOfType = EntityClasses[Entity_info[entity][0]['entitytype']]
        TypeOfEntityAttribute = type(Entity_info[entity][0][attribute])
        # Can't have list as a type in owl so need to get the type of the list elements instead
        if attribute not in EntityAttributeType.keys():
            EntityAttributeType[attribute] = {'domain':  {ClassOfType}, 'range': {TypeOfEntityAttribute}}
        else:
            EntityAttributeType[attribute]['domain'].add(ClassOfType)
            EntityAttributeType[attribute]['range'].add(TypeOfEntityAttribute)

AttributesToBeRemoved = list()
for attribute in EntityAttributeType.keys():
    EntityAttributeType[attribute]['domain'] = list(EntityAttributeType[attribute]['domain'])
    EntityAttributeType[attribute]['range'] = list(EntityAttributeType[attribute]['range'])
    if any(a not in [str,int,float,bool] for a in EntityAttributeType[attribute]['range']):
           AttributesToBeRemoved.append(attribute)

for attribute in AttributesToBeRemoved:
    EntityAttributeType.pop(attribute, None)



Now we can create the classes for the attributes

In [13]:
EntityAttributeType.pop('type',None)
with onto:
    AttributeClasses = dict()
    for attribute in EntityAttributeType.keys():
        AttributeClasses[attribute] = type(attribute, (DataProperty,), EntityAttributeType[attribute])

Now we can populate. First we populate the entities. Now to populate the entities. There is one small issue here: there is an entry in the data "entity": "Disease". This conflicts the name of the one of the types, and hence of one of the EntityClasses. We have to trap for it and rename it.

In [14]:


Nodes = dict()
with onto:
    for entity in Entity_info.keys():
        NodeAttributes = Entity_info[entity][0]
        Nodes[entity] = EntityClasses[NodeAttributes['entitytype']](name=NodeAttributes['entity'])
        for k in NodeAttributes.keys():
            getattr(Nodes[entity],k).append(NodeAttributes[k])

Now add the relations

In [ ]:
with onto:
    for relation in Relation_info.keys():
        SourceNodeName = Relation_info[relation]['source_entity']
        TargetNodeName = Relation_info[relation]['target_entity']
        RelationName = Relation_info[relation]['relationship']
        if SourceNodeName in Nodes.keys() and TargetNodeName in Nodes.keys():
            getattr(Nodes[SourceNodeName],RelationName).append(Nodes[TargetNodeName])

In [16]:
onto.save('HALD.rdf')